In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    # Auto-discover archive path (Kaggle mount paths vary)
    import glob
    if not os.path.exists(input_archive):
        candidates = glob.glob('/kaggle/input/**/wheels.tar.gz', recursive=True)
        if candidates:
            input_archive = candidates[0]
            print(f'Archive found at: {input_archive}')
        else:
            # Debug dump
            for root, dirs, files in os.walk('/kaggle/input/', topdown=True):
                depth = root.replace('/kaggle/input/', '').count(os.sep)
                if depth < 4:
                    for f in files:
                        print(f'  {os.path.join(root, f)}')
                if depth >= 4:
                    dirs[:] = []
            raise FileNotFoundError(f'wheels.tar.gz not found under /kaggle/input/')

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)

        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)


In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])


def find_model_path(model_name: str = None) -> str:
    """Auto-discover model path from /kaggle/input/."""
    import glob

    if model_name:
        candidates = [
            f'/kaggle/input/models/danielhanchen/{model_name}/transformers/default/1',
            f'/kaggle/input/models/openai/{model_name}/transformers/default/1',
            f'/kaggle/input/{model_name}/transformers/default/1',
            f'/kaggle/input/{model_name}',
        ]
        for path in candidates:
            if os.path.isfile(os.path.join(path, 'config.json')):
                print(f'Model found at: {path}')
                return path

    # Auto-search: find any model with config.json + weights
    for config_path in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        parent = os.path.dirname(config_path)
        if any(glob.glob(os.path.join(parent, f'*.{ext}')) for ext in ['safetensors', 'bin']):
            print(f'Model found via search: {parent}')
            return parent

    # Debug dump
    print('ERROR: Model not found. /kaggle/input/ contents:')
    for root, dirs, files in os.walk('/kaggle/input/', topdown=True):
        depth = root.replace('/kaggle/input/', '').count(os.sep)
        if depth < 5:
            print(f'{"  " * depth}{os.path.basename(root)}/ ({len(files)} files)')
        if depth >= 5:
            dirs[:] = []

    raise RuntimeError(f'Model "{model_name}" not found under /kaggle/input/')


MODEL_PATH = find_model_path()
print(f'\nUsing model path: {MODEL_PATH}')

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import json
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    SystemContent,
    DeveloperContent,
    ReasoningEffort,
    ToolNamespaceConfig,
    Author,
    Message,
    Role,
    TextContent,
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server


In [ ]:
class CFG:

    system_prompt = (
        'You are an IMO specialist competing in a timed exam. '
        'Your goal is to find the correct integer answer (0-99999) through rigorous reasoning.\n\n'

        '# Solving Process:\n'
        '1. UNDERSTAND: Rephrase the problem. Identify givens, unknowns, constraints.\n'
        '2. PLAN: Pick the most promising approach.\n'
        '3. EXECUTE: Work methodically. Show reasoning. Use code when computation helps.\n'
        '4. VERIFY: Check your answer against all constraints. Test edge cases.\n'
        '5. IF STUCK: Try a completely different approach. Do not repeat failed strategies.\n\n'

        '# Output Format:\n'
        '- \\Vboxed{N}: Checkpoint. Save your best candidate and KEEP WORKING.\n'
        '- \\boxed{N}: Final answer. Only when confident.\n'
        'Always have a \\Vboxed{} before attempting verification.\n\n'

        '# Time Management:\n'
        '`_DEADLINE` is set in your Python environment. Check: `print(f"{_DEADLINE - _time.time():.0f}s left")`\n'
        'Automatic warnings appear in code output:\n'
        '- ~100s left: checkpoint reminder\n'
        '- ~30s left: write \\Vboxed{N} immediately\n'
        '- ~5s left: code execution blocked, answer extracted automatically\n'
        'When you see a time warning, write \\Vboxed{N} BEFORE doing anything else.\n\n'

        '# Code Strategy:\n'
        '- Before coding, estimate complexity. >10^6 ops → find a shortcut first.\n'
        '- If code times out, do NOT reduce range and retry. Find closed-form, recurrence, or DP.\n'
        '- Trust code over reasoning. If code says X and theory says Y, trust X.\n'
        '- For combinatorics: verify formula against brute-force for n=3,4,5 before extrapolating.\n'
        '- If your answer is NOT in 0-99999, you misread the problem. Re-read and reinterpret.\n'
        '- Submit your best partial result rather than retrying indefinitely.\n\n'

        '# Efficiency:\n'
        'If the answer is obvious (direct computation, well-known identity), state it with brief '
        'justification. Do not use Python for problems you can solve in 3 lines of reasoning.'
    )

    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'

        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'

        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )

    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'

        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'

        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'

        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'

        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds\n'
        '- For very large exponents (e.g. a^(n!)), use pow(base, exp, mod) or analytical methods — never materialize the full number\n\n'

        '# Code Robustness Rules:\n'
        '- Each code cell must be SELF-CONTAINED: re-import and re-define everything you need. '
        'If a previous cell errored, its definitions are LOST.\n'
        '- Avoid sympy expressions that grow beyond a few hundred characters. '
        'If an expression is exploding, switch to numerical methods (numpy) or modular arithmetic.\n'
        '- For geometry: prefer numpy coordinate models with brute-force verification '
        'over pure symbolic or direction-counting approaches.\n'
        '- For number theory / functional equations: compute small cases first (n=1..20) '
        'to find patterns before attempting a proof. Brute-force exploration often reveals the structure.\n'
        '- For combinatorics counting: always verify no duplicates — check injectivity explicitly.\n'
        '- If 5+ code cells without progress, STOP and restart with a different approach.\n'
        '- When using Python, prefer converting sympy expressions to int() before passing to Python builtins like pow(), min(), max().\n'
        '- Common import: from sympy.ntheory.modular import crt (note: sympy.crt does not exist).\n'
        '- If a code cell fails, re-import necessary libraries in the next cell.\n'
        '- sympy.solve() often times out on complex systems. For polynomial systems with > 3 unknowns, '
        'prefer numerical methods (scipy fsolve/minimize) or manual algebraic reduction.\n'
        '- Never call sympy.simplify() or sympy.expand() on expressions involving large binomials or '
        'products of many terms. Use float() first to check the numerical value.\n\n'

        '# Library Availability:\n'
        'NOT installed (do NOT import — they WILL fail): pulp, ortools, z3-solver, mip, pyscipopt, pysat, cvxpy, constraint, matplotlib, sklearn.\n'
        'Do not waste turns trying these libraries or probing with try/except.\n\n'

        'Available and recommended:\n'
        '- sympy, numpy, scipy, networkx, mpmath, math, itertools, functools, fractions, collections, decimal, heapq, random, statistics\n\n'

        'Replacements for unavailable optimization/constraint libraries:\n'
        '- Integer/Binary Linear Programming (replaces pulp, mip, ortools):\n'
        '  from scipy.optimize import milp, LinearConstraint, Bounds\n'
        '  import numpy as np\n'
        '  res = milp(c=c_vec, constraints=LinearConstraint(A, lb, ub),\n'
        '             integrality=np.ones(n), bounds=Bounds(0, 1))\n'
        '  # res.success, res.x (solution vector), res.fun (objective value)\n'
        '- Linear Programming: scipy.optimize.linprog(c, A_ub=A, b_ub=b, method="highs")\n'
        '- Assignment problems: scipy.optimize.linear_sum_assignment(cost_matrix)\n'
        '- Graph algorithms (matching, shortest path, components): import networkx as nx\n'
        '- Constraint satisfaction: encode as ILP via scipy.optimize.milp, or backtracking with pruning\n'
        '- Number theory: sympy.ntheory or random (Monte Carlo verification)\n'
        '- Geometry: fractions (exact rational) + sympy, avoid floating point\n'
        '- Game theory/DP: functools.lru_cache for memoized recursion'
    )

    served_model_name = 'gpt-oss'
    model_path = MODEL_PATH

    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    problem_timeout = 450
    rerun_timeout = 600
    reserved_per_problem = 150

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 30
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    _is_20b = 'gpt-oss-20b' in MODEL_PATH.lower()
    attempts = 72 if _is_20b else 24
    workers = 72 if _is_20b else 24
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.90
    temperature = 0.5
    # Normal distribution: mean=0.3, clipped to [0.2, 0.4], gap 0.1
    import numpy as _np
    _rng = _np.random.default_rng(42)
    _temps_72 = (_np.clip(_rng.normal(0.3, 0.05, 72), 0.2, 0.4) / 0.1).round() * 0.1
    _temps_72 = _temps_72.round(1).tolist()
    _rng2 = _np.random.default_rng(42)
    _temps_24 = (_np.clip(_rng2.normal(0.3, 0.05, 24), 0.2, 0.4) / 0.1).round() * 0.1
    _temps_24 = _temps_24.round(1).tolist()
    temp_schedule = _temps_72 if _is_20b else _temps_24
    min_p = 0.05


In [ ]:
# ProblemDB not used in no-wave1 variant
pass


In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    DB_INSTRUCTIONS = ''

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig,
                           reasoning_effort: 'ReasoningEffort' = None) -> SystemContent:

        effort = reasoning_effort if reasoning_effort is not None else ReasoningEffort.HIGH
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=effort)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self,
        system_prompt: str,
        user_prompt: str,
        tool_config: ToolNamespaceConfig,
        reasoning_effort: 'ReasoningEffort' = None
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config, reasoning_effort)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        developer_content = DeveloperContent(instructions=self.DB_INSTRUCTIONS)
        developer_message = Message.from_role_and_content(Role.DEVELOPER, developer_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, developer_message, user_message]



In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        self._start_kernel()

    def _start_kernel(self):
        """Start a fresh Jupyter kernel and initialize it."""
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self._init_environment()

    def _init_environment(self):
        """Run the standard imports in the kernel."""
        self.execute(
            'import sys\n'
            'sys.set_int_max_str_digits(100000)\n'
            'import math\n'
            'import numpy\n'
            'import numpy as np\n'
            'import sympy\n'
            'import sympy as sp\n'
            'from sympy import symbols, solve, Eq, simplify, factor, expand, Rational, oo, sqrt, pi, gcd, lcm, isprime, nextprime, factorint, totient, mod_inverse, binomial, factorial, Mod, Poly\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'from fractions import Fraction\n'
            'import random\n'
            'import time as _time\n'
            'import time\n'
            'from collections import deque\n'
            'import networkx as nx\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=False,
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                # Force-kill kernel to stop zombie C-level computations (e.g., HiGHS MIP solver)
                # interrupt_kernel() only sends SIGINT which C libraries may ignore
                self._force_restart()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def _force_restart(self):
        """Kill the current kernel and start a fresh one. Handles zombie C-level computations."""
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()
        with contextlib.suppress(Exception):
            if self._km:
                self._km.shutdown_kernel(now=True)
        with contextlib.suppress(Exception):
            if self._km:
                self._km.cleanup_resources()
        # Start fresh
        self._start_kernel()

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute(
            '%reset -f\n'
            'import sys\n'
            'sys.set_int_max_str_digits(100000)\n'
            'import math\n'
            'import numpy\n'
            'import numpy as np\n'
            'import sympy\n'
            'import sympy as sp\n'
            'from sympy import symbols, solve, Eq, simplify, factor, expand, Rational, oo, sqrt, pi, gcd, lcm, isprime, nextprime, factorint, totient, mod_inverse, binomial, factorial, Mod, Poly\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'from fractions import Fraction\n'
            'import random\n'
            'import time as _time\n'
            'import time\n'
            'from collections import deque\n'
            'import networkx as nx\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()


In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        # Skip wrapping assignments (x = 5 → print(x = 5) causes TypeError)
        # Also skip augmented assignments (+=, -=, etc.), for/while/if/else/with/try/class/def, and decorators
        if (re.search(r'(?<![=!<>])=(?!=)', last_line)
                or last_line.split()[0] in ('for', 'while', 'if', 'elif', 'else:', 'with', 'try:', 'except', 'except:', 'finally:', 'class', 'def', 'return', 'raise', 'del', 'assert', 'pass', 'break', 'continue')
                or last_line.startswith('@')):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        # Strip markdown code fences if model wraps code in them
        stripped = raw_script.strip()
        if stripped.startswith('```'):
            lines = raw_script.split('\n')
            if lines[0].strip().startswith('```'):
                lines = lines[1:]
            if lines and lines[-1].strip() == '```':
                lines = lines[:-1]
            raw_script = '\n'.join(lines)
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        # Cap output length to prevent notebook bloat (HiGHS MILP can dump 10K+ lines)
        max_output_chars = 8000
        if len(output) > max_output_chars:
            truncated = len(output) - max_output_chars
            output = output[:max_output_chars // 2] + f'\n... [{truncated} chars truncated] ...\n' + output[-max_output_chars // 2:]

        return [self._make_response(output, channel=message.channel)]

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()

        # Preload GenSelect model (Nemotron-7B BNB-4bit on GPU) while vLLM warms up
        self._load_genselect_model()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.port),
            '--dtype',
            self.cfg.dtype,
            '--kv-cache-dtype',
            self.cfg.kv_cache_dtype,
            '--max-model-len',
            str(self.cfg.context_tokens),
            '--stream-interval',
            str(self.cfg.stream_interval),
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox_with_retry(max_retries=3):
            for attempt in range(max_retries):
                try:
                    return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
                except Exception as e:
                    if attempt < max_retries - 1:
                        time.sleep(1)
                    else:
                        raise RuntimeError(f'Kernel init failed after {max_retries} retries: {e}')

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox_with_retry) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> tuple[int | None, float]:
        """Extract answer from text. Returns (answer, confidence).
        confidence=1.0 for \\boxed{}, 0.7 for \\Vboxed{} or fallback patterns."""

        def _try_extract(val_str: str) -> int | None:
            try:
                clean = val_str.replace(',', '').strip()
                val = int(clean)
                if 0 <= val <= 99999:
                    return val
            except ValueError:
                pass
            return None

        # Priority 1: \boxed{N} — full confidence (last match wins)
        matches = re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 1.0

        # Priority 2: \Vboxed{N} — checkpoint, reduced confidence (last match wins)
        matches = re.findall(r'\\Vboxed\s*\{\s*([0-9,]+)\s*\}', text)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 3: 'final answer is X'
        matches = re.findall(r'final\s+answer\s+is\s*([0-9,]+)', text, re.IGNORECASE)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 4: 'the answer is X' / 'answer is X'
        matches = re.findall(r'(?:the\s+)?answer\s+is\s*:?\s*\*?\*?\s*([0-9,]+)', text, re.IGNORECASE)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 5: 'answer: X' / 'Answer = X'
        matches = re.findall(r'answer\s*[:=]\s*\*?\*?\s*([0-9,]+)', text, re.IGNORECASE)
        if matches:
            val = _try_extract(matches[-1])
            if val is not None:
                return val, 0.7

        # Priority 6: Any numeric content inside \boxed{} (handles LaTeX formatting)
        matches = re.findall(r'\\boxed\s*\{([^}]+)\}', text)
        if matches:
            nums = re.findall(r'(\d+)', matches[-1])
            if nums:
                val = _try_extract(nums[-1])
                if val is not None:
                    return val, 0.5

        return None, 0.0


    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float,
        temperature: float = None,
        flush_problem: str = None,
        reasoning_effort: 'ReasoningEffort' = None
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'Temperature': temperature,
                'Confidence': 0.0,
                'Source': 'skipped',
                'Stop_Reason': 'early_skip',
                'Successful_Code': 0,
                'Conversation': [],
                'Time': 0.0
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        answer_confidence = 0.0
        answer_source = 'none'
        
        logprobs_buffer = []
        conversation_log = []
        vboxed_checkpoints = []
        bells = {'notice': 0, 'urgent': 0, 'code_blocked': 0}
        stop_reason = 'max_turns'
        turn_num = 0
        consecutive_errors = 0
        attempt_start = time.time()
    
        attempt_seed = (self.cfg.seed + attempt_index) ** 2
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            # Inject deadline into sandbox so model can check time remaining
            sandbox.execute(f'import time as _time; _DEADLINE = {deadline}')
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt,
                problem,
                local_tool.tool_config,
                reasoning_effort=reasoning_effort
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                turn_num += 1
                time_left = deadline - time.time()
                if stop_event.is_set() or time_left <= 0:
                    break

                # Hard takeover at ≤5s: scan conversation for any answer, or force emergency completion
                if time_left <= 5 and final_answer is None:
                    # Try to salvage an answer from existing conversation
                    if conversation_log:
                        full_text = ' '.join(
                            entry.get('text', '') + ' ' + entry.get('code', '') + ' ' + entry.get('output', '')
                            for entry in conversation_log
                        )
                        cand, conf = self._scan_for_answer(full_text)
                        if cand is not None:
                            final_answer, answer_confidence = cand, conf
                            answer_source = 'emergency_checkpoint'
                            stop_reason = 'emergency_deadline'
                            break

                    # Emergency: tiny completion with problem context to force a \Vboxed{N}
                    _flush_problem = flush_problem or problem
                    flush_msgs = self.template.apply_chat_template(
                        system_prompt,
                        f'PROBLEM: {_flush_problem}\n\nYou are out of time. Output ONLY \\Vboxed{{N}} where N is your best integer answer (0-99999). No other text.',
                        local_tool.tool_config,
                        reasoning_effort=reasoning_effort
                    )
                    flush_conv = Conversation.from_messages(flush_msgs)
                    flush_ids = encoding.render_conversation_for_completion(flush_conv, Role.ASSISTANT)
                    flush = self.client.completions.create(
                        model=self.cfg.served_model_name,
                        temperature=0.0,
                        max_tokens=64,
                        prompt=flush_ids,
                        seed=attempt_seed,
                        stream=False,
                        extra_body={'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                    )
                    flush_text = flush.choices[0].text or ''
                    cand, conf = self._scan_for_answer(flush_text)
                    if cand is not None:
                        final_answer, answer_confidence = cand, conf
                        answer_source = 'emergency_forced'
                        stop_reason = 'emergency_deadline'
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=(temperature if temperature is not None else self.cfg.temperature), 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer, conf = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                answer_confidence = conf
                                answer_source = 'streaming'
                                if conf >= 1.0:
                                    break  # only stop on \boxed{}, let \Vboxed{} continue
                                else:
                                    vboxed_checkpoints.append({'turn': turn_num, 'answer': answer})

                finally:
                    stream.close()
    
                if final_answer is not None and answer_confidence >= 1.0:
                    break

                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                # Log reasoning text (code calls are logged below with their output)
                msg_text = last_message.content[0].text if last_message.content else ''
                if last_message.recipient != 'python':
                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'reasoning',
                        'text': msg_text,
                        'code': '',
                        'output': '',
                        'error': False
                    })
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer, answer_confidence = self._scan_for_answer(answer_text)
                    answer_source = 'final_channel'
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    raw_code = last_message.content[0].text

                    # Block code execution in last 5s — force checkpoint instead
                    time_left = deadline - time.time()
                    if time_left <= 5:
                        bells['code_blocked'] += 1
                        tool_responses = [local_tool._make_response(
                            f'[DEADLINE] {time_left:.0f}s left. Code execution blocked. Write \\Vboxed{{N}} with your best answer NOW.',
                            channel=last_message.channel
                        )]
                    else:
                        tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'code_call',
                        'code': raw_code,
                        'output': response_text,
                        'error': response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text
                    })
    
                    is_error = response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text
                    if is_error:
                        python_errors += 1
                        consecutive_errors += 1
                        if consecutive_errors >= 8:
                            stop_reason = 'error_cascade'
                            break
                    else:
                        consecutive_errors = 0

                    # Exam bell: 3-tier time warnings in tool responses
                    # Tier 1 (<100s): heads up, start wrapping up
                    # Tier 2 (<30s): must Vboxed now, keep working but checkpoint first
                    # Tier 3 (<5s): hard takeover (handled at top of turn loop)
                    time_remaining = deadline - time.time()
                    if time_remaining < 30:
                        bells['urgent'] += 1
                        bell = '\n\n[URGENT — <30s remaining. You MUST write \\Vboxed{YOUR_ANSWER} with your best answer RIGHT NOW before continuing.]'
                        tool_responses = [local_tool._make_response(response_text + bell, channel=last_message.channel)]
                    elif time_remaining < 100:
                        bells['notice'] += 1
                        bell = f'\n\n[NOTICE — ~{time_remaining:.0f}s remaining on this problem.]'
                        tool_responses = [local_tool._make_response(response_text + bell, channel=last_message.channel)]

                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
            stop_reason = f'exception:{type(exc).__name__}'
    
        finally:
            if sandbox is not None:
                try:
                    sandbox.reset()
                    self.sandbox_pool.put(sandbox)
                except Exception:
                    try:
                        sandbox.close()
                    except Exception:
                        pass
                    try:
                        self.sandbox_pool.put(AIMO3Sandbox(timeout=self.cfg.jupyter_timeout))
                    except Exception:
                        pass
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)

        attempt_elapsed = time.time() - attempt_start
        time_remaining = deadline - time.time()

        # If no answer found via streaming, scan full conversation for Vboxed checkpoints
        if final_answer is None and conversation_log:
            full_text = ' '.join(entry.get('text', '') + ' ' + entry.get('code', '') + ' ' + entry.get('output', '') for entry in conversation_log)
            final_answer, answer_confidence = self._scan_for_answer(full_text)
            answer_source = 'fallback'

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'Temperature': temperature,
            'Answer': final_answer,
            'Confidence': answer_confidence,
            'Source': answer_source,
            'Stop_Reason': stop_reason,
            'Successful_Code': python_calls - python_errors,
            'Conversation': conversation_log,
            'Vboxed_Checkpoints': vboxed_checkpoints,
            'Bells': bells,
            'Time_Remaining': round(time_remaining, 1),
            'Time': round(attempt_elapsed, 1)
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            successful_code = result.get('Successful_Code', 0)

            if answer is not None:
                # Base weight: inverse entropy
                if not math.isfinite(entropy) or entropy <= 0:
                    entropy = float('inf')
                weight = 1.0 / max(entropy, 1e-9)

                # Downweight attempts with zero successful code executions
                if successful_code == 0:
                    weight *= 0.5

                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight
            })

        # Deterministic tie-breaking: votes (desc), then score (desc), then answer value (desc)
        scored_answers.sort(key=lambda x: (x['votes'], x['score'], x['answer']), reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'],
                item['votes'],
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data,
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        print(vote_dataframe.to_string(index=False))

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0, 0

        final_answer = scored_answers[0]['answer']
        top_votes = scored_answers[0]['votes']
        print(f'\nFinal Answer: {final_answer} (votes: {top_votes})\n')

        return final_answer, top_votes

    def solve_problem(self, problem: str) -> int:

        print(f'\nProblem: {problem}\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'

        # Log the full prompt sent to model
        print(f'\n=== USER PROMPT ({len(user_input)} chars) ===')
        print(user_input)
        print(f'=== END USER PROMPT ===\n')

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global

        reserved = max(0, self.problems_remaining - 1) * self.cfg.reserved_per_problem
        budget = time_left - reserved
        budget = min(budget, self.cfg.problem_timeout)
        budget = max(budget, 30)

        # Hard cutoff: if truly out of time, skip
        if time_left <= 0:
            print(f'Out of time (elapsed={elapsed_global:.0f}s > limit={self.cfg.notebook_limit}s) — returning 0')
            self.problems_remaining = max(0, self.problems_remaining - 1)
            return 0

        deadline = time.time() + budget

        effort = ReasoningEffort.HIGH
        effort_reason = 'always-high'
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f} | TimeLeft: {time_left:.0f}s | Effort: {effort_reason}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            temp = self.cfg.temp_schedule[attempt_index] if hasattr(self.cfg, 'temp_schedule') and self.cfg.temp_schedule else self.cfg.temperature
            tasks.append((self.cfg.system_prompt, attempt_index, temp))
    
        detailed_results = []
        valid_answers = []
        # Streaming summarization: summarize each attempt as it finishes during R1.
        # Summaries overlap with remaining R1 work — practically free.
        summary_futures = {}  # attempt_index → future
        summary_results = {}  # attempt_index → summary_text
        summary_executor = ThreadPoolExecutor(max_workers=2)  # 2 concurrent — minimize vLLM contention with R1

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index, temp) in tasks:
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    system_prompt,
                    attempt_index,
                    stop_event,
                    deadline,
                    temp,
                    None,  # flush_problem
                    effort
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                        # Stream: immediately submit for summarization
                        att_idx = result.get('Attempt', len(detailed_results))
                        n_conv = len(result.get('Conversation', []))
                        print(f'  → Att {att_idx} finished (answer={result["Answer"]}, '
                              f'{n_conv} turns) — submitting for summarization')
                        sfut = summary_executor.submit(
                            self._summarize_one_attempt, problem, result
                        )
                        summary_futures[att_idx] = sfut

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)

        # Collect all streaming summaries (most should already be done)
        r1_end_time = time.time()
        r1_elapsed = r1_end_time - (deadline - budget) if budget else 0
        print(f'\n=== STREAMING SUMMARIES: collecting {len(summary_futures)} summaries ===')
        print(f'R1 finished in {r1_elapsed:.0f}s | {len(valid_answers)} valid answers | '
              f'{len(summary_futures)} summaries submitted')
        if summary_futures:
            summ_wait_start = time.time()
            n_already_done = sum(1 for sfut in summary_futures.values() if sfut.done())
            print(f'  {n_already_done}/{len(summary_futures)} already done before collection')
            for att_idx, sfut in summary_futures.items():
                try:
                    summary_results[att_idx] = sfut.result(timeout=45)
                except Exception as e:
                    print(f'  Summary Att {att_idx}: failed — {type(e).__name__}: {e}')
            summary_executor.shutdown(wait=False)
            summ_wait = time.time() - summ_wait_start

            # Detailed summary stats
            n_ok = sum(1 for v in summary_results.values() if v.get('status') == 'ok')
            n_mismatch = sum(1 for v in summary_results.values() if v.get('status') == 'answer-mismatch')
            n_short = sum(1 for v in summary_results.values() if v.get('status') == 'skip-short')
            n_err = sum(1 for v in summary_results.values() if 'error' in str(v.get('status', '')))
            avg_time = sum(v.get('time', 0) for v in summary_results.values()) / max(len(summary_results), 1)
            avg_compression = 0
            compressions = []
            for v in summary_results.values():
                tc = v.get('trace_chars', 0)
                sc = v.get('summary_chars', 0)
                if tc > 0 and sc > 0:
                    compressions.append(sc / tc)
            if compressions:
                avg_compression = sum(compressions) / len(compressions)

            print(f'Summaries: {len(summary_results)}/{len(summary_futures)} collected '
                  f'(waited {summ_wait:.1f}s)')
            print(f'  Status: {n_ok} ok, {n_mismatch} mismatch, {n_short} skip-short, {n_err} error')
            print(f'  Avg summarization time: {avg_time:.1f}s | Avg compression ratio: {avg_compression:.2f}')
        else:
            print(f'  No summaries (no valid answers)')
        print(f'=== END STREAMING SUMMARIES ===\n')
        self._cached_summaries = summary_results

        self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')

            display_cols = [c for c in results_dataframe.columns if c not in ('Conversation', 'Vboxed_Checkpoints', 'Bells')]
            print(results_dataframe[display_cols].to_string(index=False))

        self._last_detailed_results = detailed_results

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        answer, top_votes = self._select_answer(detailed_results)
        r1_answer = answer  # save for logging
        all_results = detailed_results

        # Count competing answer clusters for logging
        from collections import Counter as _AnswerCounter
        _ans_counts = _AnswerCounter(r['Answer'] for r in detailed_results if r['Answer'] is not None)
        _num_clusters = len(_ans_counts)

        # --- Dual GenSelect: 7B trained + 120B zero-shot in parallel ---
        gs_phase_start = time.time()
        _gs_elapsed = time.time() - self.notebook_start_time
        _gs_time_left = self.cfg.notebook_limit - _gs_elapsed
        print(f'\n=== GENSELECT PHASE: launching dual judges ===')
        print(f'R1 answer: {answer} (votes: {top_votes}/{len(valid_answers)})')
        print(f'Competing clusters: {_num_clusters} ({dict(_ans_counts.most_common())})')
        print(f'Time elapsed: {_gs_elapsed:.0f}s | Time left: {_gs_time_left:.0f}s')

        gs_executor = ThreadPoolExecutor(max_workers=2)
        gs7b_fut = gs_executor.submit(self._run_genselect, problem, all_results)
        gs120b_fut = gs_executor.submit(self._run_genselect_vllm, problem, all_results)
        gs7b_answer, gs7b_reasoning = gs7b_fut.result()
        gs120b_answer, gs120b_reasoning = gs120b_fut.result()
        gs_executor.shutdown(wait=False)
        gs_phase_elapsed = time.time() - gs_phase_start

        # 3-way decision: majority vote + 7B judgment + 120B judgment
        judges = [('majority', answer), ('GenSelect-7B', gs7b_answer), ('GenSelect-120B', gs120b_answer)]
        valid_judges = [(name, ans) for name, ans in judges if ans is not None]

        print(f'\n=== 3-WAY DECISION ===')
        print(f'  majority:       {answer} (votes: {top_votes}/{len(valid_answers)})')
        print(f'  GenSelect-7B:   {gs7b_answer} (reasoning: {len(gs7b_reasoning) if gs7b_reasoning else 0} chars)')
        print(f'  GenSelect-120B: {gs120b_answer} (reasoning: {len(gs120b_reasoning) if gs120b_reasoning else 0} chars)')
        print(f'  GenSelect phase time: {gs_phase_elapsed:.1f}s')

        gs_disagrees = False
        rerun_reason = []
        rerun_threshold = math.ceil(self.cfg.attempts / 3)

        if len(valid_judges) >= 2:
            jvotes = _AnswerCounter(ans for _, ans in valid_judges)
            consensus_answer, consensus_count = jvotes.most_common(1)[0]
            gs_only = [(n, a) for n, a in valid_judges if n != 'majority']

            if consensus_count >= 2:
                # 2+ judges agree — use consensus
                if consensus_answer != answer:
                    print(f'  Verdict: OVERRIDE — consensus {consensus_answer} overrides majority {answer}')
                    answer = consensus_answer
                else:
                    print(f'  Verdict: CONFIRMED — consensus matches majority {answer}')
            elif len(valid_judges) == 3:
                # All 3 disagree (each has count 1)
                print(f'  Verdict: ALL 3 DISAGREE — triggering rerun')
                gs_disagrees = True
                rerun_reason.append('all 3 judges disagree')
            else:
                # 2 valid judges, no consensus (majority + 1 GenSelect that disagrees)
                gs_name, gs_ans = gs_only[0] if gs_only else ('?', None)
                print(f'  Verdict: SPLIT — {gs_name} says {gs_ans} vs majority {answer} — triggering rerun')
                gs_disagrees = True
                rerun_reason.append(f'{gs_name} disagrees ({gs_ans} vs majority {answer})')
        else:
            # Only majority (both GenSelects returned None or skipped) — trust majority
            print(f'  Verdict: SOLE JUDGE — GenSelects inconclusive ({len(valid_judges)} valid), trusting majority {answer}')

        print(f'=== END 3-WAY DECISION ===')

        if top_votes < rerun_threshold:
            rerun_reason.append(f'low confidence ({top_votes}/{rerun_threshold})')

        should_rerun = (top_votes < rerun_threshold) or gs_disagrees
        print(f'\n=== RERUN CHECK: top_votes={top_votes}, threshold={rerun_threshold}, gs_disagrees={gs_disagrees}, should_rerun={should_rerun} ===')

        if should_rerun:
            elapsed_global = time.time() - self.notebook_start_time
            time_left = self.cfg.notebook_limit - elapsed_global
            reserved = max(0, self.problems_remaining) * self.cfg.reserved_per_problem
            rerun_budget = time_left - reserved
            rerun_budget = min(rerun_budget, self.cfg.rerun_timeout)

            if rerun_budget > self.cfg.reserved_per_problem:
                print(f'\nRerun triggered: {", ".join(rerun_reason)}. Budget: {rerun_budget:.0f}s\n')
                rerun_deadline = time.time() + rerun_budget

                rerun_context = self._build_rerun_context(detailed_results)

                # Append dual GenSelect expert opinions
                gs_opinions = []
                if gs7b_answer is not None:
                    gs_opinions.append(f'Trained judge (7B): answer={gs7b_answer}')
                    if gs7b_reasoning:
                        gs_opinions.append(f'  Reasoning: {gs7b_reasoning[:300]}')
                if gs120b_answer is not None:
                    gs_opinions.append(f'Zero-shot judge (120B): answer={gs120b_answer}')
                    if gs120b_reasoning:
                        gs_opinions.append(f'  Reasoning: {gs120b_reasoning[:300]}')

                if gs_opinions:
                    rerun_context += (
                        f'\n\n=== EXPERT ANALYSIS (independent mathematical judges) ===\n'
                        + '\n'.join(gs_opinions)
                        + f'\nThese judges may be wrong — carefully verify with your own computation.\n'
                        f'=== END EXPERT ANALYSIS ===\n'
                    )
                    print(f'Rerun context: {len(rerun_context)} chars (~{len(rerun_context)//4} tokens) | GenSelect opinions: {len(gs_opinions)} lines')

                rerun_input = rerun_context + '\n\n' + user_input

                rerun_results = []
                stop_event2 = threading.Event()
                executor2 = ThreadPoolExecutor(max_workers=self.cfg.workers)

                try:
                    futures2 = []
                    for attempt_index in range(self.cfg.attempts):
                        temp = self.cfg.temp_schedule[attempt_index] if hasattr(self.cfg, 'temp_schedule') and self.cfg.temp_schedule else self.cfg.temperature
                        future = executor2.submit(
                            self._process_attempt,
                            rerun_input,
                            self.cfg.system_prompt,
                            attempt_index + self.cfg.attempts,
                            stop_event2,
                            rerun_deadline,
                            temp,
                            user_input,  # flush_problem
                            ReasoningEffort.HIGH  # always HIGH for reruns
                        )
                        futures2.append(future)

                    for future in as_completed(futures2):
                        try:
                            result = future.result()
                            rerun_results.append(result)
                        except Exception as exc:
                            print(f'Rerun future failed: {exc}')

                finally:
                    stop_event2.set()
                    executor2.shutdown(wait=True, cancel_futures=True)

                # Merge all results and re-vote
                all_results = detailed_results + rerun_results
                self._last_detailed_results = all_results
                print(f'\nRerun complete: {len(rerun_results)} new results, {len(all_results)} total')
                answer, top_votes = self._select_answer(all_results)

                # Final dual GenSelect on combined R1+R2 results
                gs2_phase_start = time.time()
                print(f'\n=== FINAL GENSELECT PHASE: judging {len(all_results)} combined R1+R2 results ===')
                gs2_executor = ThreadPoolExecutor(max_workers=2)
                gs2_7b_fut = gs2_executor.submit(self._run_genselect, problem, all_results)
                gs2_120b_fut = gs2_executor.submit(self._run_genselect_vllm, problem, all_results)
                gs2_7b_answer, gs2_7b_reasoning = gs2_7b_fut.result()
                gs2_120b_answer, gs2_120b_reasoning = gs2_120b_fut.result()
                gs2_executor.shutdown(wait=False)
                gs2_phase_elapsed = time.time() - gs2_phase_start

                # Final 3-way decision
                judges2 = [('majority', answer), ('GenSelect-7B', gs2_7b_answer), ('GenSelect-120B', gs2_120b_answer)]
                valid_judges2 = [(n, a) for n, a in judges2 if a is not None]

                print(f'\n=== FINAL 3-WAY DECISION (post-rerun) ===')
                print(f'  majority:       {answer} (votes: {top_votes})')
                print(f'  GenSelect-7B:   {gs2_7b_answer} (reasoning: {len(gs2_7b_reasoning) if gs2_7b_reasoning else 0} chars)')
                print(f'  GenSelect-120B: {gs2_120b_answer} (reasoning: {len(gs2_120b_reasoning) if gs2_120b_reasoning else 0} chars)')
                print(f'  GenSelect phase time: {gs2_phase_elapsed:.1f}s')

                if len(valid_judges2) >= 2:
                    jvotes2 = _AnswerCounter(a for _, a in valid_judges2)
                    final_consensus, final_count = jvotes2.most_common(1)[0]
                    if final_count >= 2 and final_consensus != answer:
                        print(f'  Verdict: FINAL OVERRIDE — {answer} → {final_consensus}')
                        answer = final_consensus
                    elif final_count >= 2:
                        print(f'  Verdict: FINAL CONFIRMED — {answer}')
                    else:
                        # All 3 disagree post-rerun: can't rerun again, defer to 7B trained judge
                        # (it's specifically trained for solution selection)
                        if gs2_7b_answer is not None:
                            print(f'  Verdict: FINAL NO CONSENSUS — deferring to trained 7B: {gs2_7b_answer} (majority was {answer})')
                            answer = gs2_7b_answer
                        elif gs2_120b_answer is not None:
                            print(f'  Verdict: FINAL NO CONSENSUS — 7B unavailable, using 120B: {gs2_120b_answer} (majority was {answer})')
                            answer = gs2_120b_answer
                        else:
                            print(f'  Verdict: FINAL NO CONSENSUS — both judges unavailable, keeping majority {answer}')
                else:
                    print(f'  Verdict: FINAL INCONCLUSIVE — GenSelects returned None, keeping majority {answer}')
                print(f'=== END FINAL 3-WAY DECISION ===')
            else:
                print(f'Rerun skipped — insufficient budget ({rerun_budget:.0f}s < {self.cfg.reserved_per_problem}s reserved)')
                # No budget: if GenSelect consensus overrides, use it
                gs_answers = [a for a in [gs7b_answer, gs120b_answer] if a is not None and a != answer]
                if gs_answers and top_votes < rerun_threshold:
                    gc = _AnswerCounter(gs_answers)
                    gs_consensus, gc_count = gc.most_common(1)[0]
                    if gc_count >= 2 or (gc_count == 1 and len(gs_answers) == 1):
                        print(f'Low confidence + no budget — using GenSelect answer {gs_consensus} (was {answer})')
                        answer = gs_consensus

        # Final answer summary
        problem_elapsed = time.time() - (deadline - budget)
        print(f'\n=== PROBLEM COMPLETE ===')
        path_parts = [f'R1={r1_answer}(v{top_votes})']
        if gs7b_answer is not None or gs120b_answer is not None:
            path_parts.append(f'GS=7B:{gs7b_answer}/120B:{gs120b_answer}')
        did_rerun = should_rerun and rerun_budget > self.cfg.reserved_per_problem
        if did_rerun:
            path_parts.append(f'RERUN({len(rerun_results)}atts)')
            gs2_7b = gs2_7b_answer if 'gs2_7b_answer' in dir() else None
            gs2_120b = gs2_120b_answer if 'gs2_120b_answer' in dir() else None
            if gs2_7b is not None or gs2_120b is not None:
                path_parts.append(f'GS2=7B:{gs2_7b}/120B:{gs2_120b}')
        print(f'Final answer: {answer}')
        print(f'Path: {" -> ".join(path_parts)} -> FINAL={answer}')
        print(f'Total time: {problem_elapsed:.1f}s')
        # Summarization stats
        cached = getattr(self, '_cached_summaries', {})
        if cached:
            s_ok = sum(1 for v in cached.values() if v.get('status') == 'ok')
            s_total = len(cached)
            s_total_time = sum(v.get('time', 0) for v in cached.values())
            print(f'Summaries: {s_ok}/{s_total} clean | total summ time: {s_total_time:.1f}s (overlapped with R1)')
        _final_elapsed = time.time() - self.notebook_start_time
        _final_left = self.cfg.notebook_limit - _final_elapsed
        print(f'Session: {_final_elapsed:.0f}s elapsed, {_final_left:.0f}s remaining, {self.problems_remaining} problems left')
        print(f'=== END PROBLEM ===\n')

        # Store problem metadata for summary.txt
        cached = getattr(self, '_cached_summaries', {})
        s_ok = sum(1 for v in cached.values() if v.get('status') == 'ok')
        s_mismatch = sum(1 for v in cached.values() if v.get('status') == 'answer-mismatch')
        s_total = len(cached)
        s_total_time = sum(v.get('time', 0) for v in cached.values())
        self._last_problem_meta = {
            'r1_answer': r1_answer,
            'r1_votes': top_votes,
            'r1_clusters': _num_clusters,
            'gs7b': gs7b_answer,
            'gs120b': gs120b_answer,
            'gs_time': gs_phase_elapsed,
            'rerun': did_rerun if 'did_rerun' in dir() else False,
            'rerun_atts': len(rerun_results) if 'rerun_results' in dir() else 0,
            'gs2_7b': gs2_7b_answer if 'gs2_7b_answer' in dir() else None,
            'gs2_120b': gs2_120b_answer if 'gs2_120b_answer' in dir() else None,
            'summ_ok': s_ok,
            'summ_mismatch': s_mismatch,
            'summ_total': s_total,
            'summ_time': round(s_total_time, 1),
            'path': ' -> '.join(path_parts) + f' -> FINAL={answer}',
            'final_answer': answer,
        }

        return answer

    def _summarize_attempt(self, result: dict) -> dict:
        """Extract structured signal from one attempt's conversation."""
        import re
        conv = result.get('Conversation', [])

        # 1. Libraries/methods used (from code imports)
        libs = set()
        for entry in conv:
            if entry.get('type') == 'code_call':
                code = entry.get('code', '')
                for m in re.findall(r'(?:from|import)\s+([\w.]+)', code):
                    root = m.split('.')[0]
                    if root not in ('sys', 'os', 'time', 'typing', 'json'):
                        libs.add(root)

        # 2. Vboxed checkpoints — intermediate answers tried
        checkpoints = result.get('Vboxed_Checkpoints', [])

        # 3. Errors — first unique error message
        errors = []
        for entry in conv:
            if entry.get('error'):
                out = entry.get('output', '').strip()
                # Extract just the error type + message
                err_match = re.search(r'(\w*Error:\s*.{0,100})', out)
                if err_match:
                    errors.append(err_match.group(1))
                elif 'Timeout' in out or 'timeout' in out:
                    errors.append('Timeout')

        # 4. Code outputs that produced numbers (likely computed answers)
        computed_values = []
        for entry in conv:
            if entry.get('type') == 'code_call' and not entry.get('error'):
                out = entry.get('output', '').strip()
                # Find printed numbers in output
                nums = re.findall(r'\b(\d{1,5})\b', out[-200:] if len(out) > 200 else out)
                for n in nums[-3:]:  # last few numbers are most relevant
                    val = int(n)
                    if 0 < val < 100000 and val not in computed_values:
                        computed_values.append(val)

        return {
            'libs': sorted(libs)[:6],
            'checkpoints': checkpoints[-5:] if checkpoints else [],
            'errors': list(dict.fromkeys(errors))[:2],
            'computed': computed_values[-5:],
        }

    def _build_cluster_summaries(self, detailed_results: list) -> str:
        """Build per-answer-cluster summaries with approach, key code, and cross-cluster signals."""
        from collections import Counter

        # Group by answer
        clusters = {}
        for r in detailed_results:
            key = r['Answer']
            if key not in clusters:
                clusters[key] = []
            clusters[key].append(r)

        # All non-None answers for cross-cluster detection
        all_answers = set(a for a in clusters if a is not None)

        # Sort clusters by size descending, skip singletons
        sorted_clusters = sorted(
            ((ans, atts) for ans, atts in clusters.items() if len(atts) >= 2),
            key=lambda x: -len(x[1])
        )

        parts = []
        total_chars = 0
        MAX_CHARS = 1500

        for ans, atts in sorted_clusters:
            if total_chars >= MAX_CHARS:
                break

            if ans is None:
                # None cluster: count + most common stop reason
                stops = Counter(r.get('Stop_Reason', '?') for r in atts)
                top_stop = stops.most_common(1)[0][0] if stops else '?'
                part = f'{len(atts)}x returned None (common: {top_stop})'
                parts.append(part)
                total_chars += len(part)
                continue

            # Pick best representative: most successful code, then most turns
            def _rep_score(r):
                conv = r.get('Conversation', [])
                successful = sum(1 for e in conv if e.get('type') == 'code_call' and not e.get('error'))
                return (successful, len(conv))
            best = max(atts, key=_rep_score)
            conv = best.get('Conversation', [])
            n_turns = len(conv)
            time_s = best.get('Time', 0)

            # Extract approach: first reasoning text >30 chars
            approach = ''
            for entry in conv:
                if entry.get('type') == 'reasoning' and len(entry.get('text', '')) > 30:
                    approach = entry['text'].strip().replace('\n', ' ')[:200]
                    break

            # Extract key computation: last successful code + output
            key_code = ''
            key_output = ''
            for entry in reversed(conv):
                if entry.get('type') == 'code_call' and not entry.get('error') and entry.get('output', '').strip():
                    key_code = entry['code'].strip().replace('\n', '; ')[:200]
                    key_output = entry['output'].strip().replace('\n', ' ')[:150]
                    break

            # Cross-cluster signal: did any attempt in this cluster compute a value matching another cluster's answer?
            cross_signals = []
            other_answers = all_answers - {ans}
            for r in atts:
                s = self._summarize_attempt(r)
                for val in s.get('computed', []):
                    if val in other_answers:
                        cross_signals.append((r['Attempt'], val))
            # Deduplicate by value
            seen_vals = set()
            unique_cross = []
            for att_num, val in cross_signals:
                if val not in seen_vals:
                    seen_vals.add(val)
                    unique_cross.append((att_num, val))

            # Build cluster text
            header = f'{len(atts)}x answered {ans} (best: Att {best["Attempt"]}, {n_turns} turns, {time_s:.0f}s)'
            lines = [header]
            if approach:
                lines.append(f'  Approach: {approach}')
            if key_code:
                lines.append(f'  Code: {key_code}')
            if key_output:
                lines.append(f'  Output: {key_output}')
            for att_num, val in unique_cross[:2]:
                lines.append(f'  NOTE: Att {att_num} also computed {val} (matches another cluster)')

            part = '\n'.join(lines)
            if total_chars + len(part) > MAX_CHARS and parts:
                break
            parts.append(part)
            total_chars += len(part)

        if not parts:
            return ''

        return '--- Answer clusters ---\n' + '\n\n'.join(parts) + '\n--- End clusters ---'

    def _build_rerun_context(self, detailed_results: list) -> str:
        """Build context from round 1 results for rerun attempts."""
        from collections import Counter

        answer_counts = Counter()
        none_count = 0
        attempt_lines = []

        for r in sorted(detailed_results, key=lambda x: x['Attempt']):
            ans = r['Answer']
            temp = r.get('Temperature', '?')
            code_calls = r.get('Python Calls', 0)
            errors = r.get('Python Errors', 0)
            stop = r.get('Stop_Reason', '')

            if ans is None:
                none_count += 1
                status = f'None ({stop})'
            else:
                answer_counts[ans] += 1
                status = str(ans)

            summary = self._summarize_attempt(r)

            time_s = r.get('Time', 0)
            turns = len(r.get('Conversation', []))
            line = f'  Att {r["Attempt"]:>2} (t={temp}): {status} | {time_s:.0f}s {turns}turns code={code_calls} err={errors}'
            if summary['libs']:
                line += f' | libs: {", ".join(summary["libs"])}'
            if summary['checkpoints']:
                line += f' | checkpoints: {summary["checkpoints"]}'
            if summary['computed']:
                line += f' | computed: {summary["computed"]}'
            if summary['errors']:
                line += f' | errors: {"; ".join(summary["errors"])}'
            # Skip 0-turn attempts in per-attempt detail (no signal)
            if turns > 0:
                attempt_lines.append(line)

        # Vote summary
        vote_parts = []
        for ans, cnt in answer_counts.most_common():
            vote_parts.append(f'{ans} x{cnt}')
        if none_count:
            vote_parts.append(f'None x{none_count}')
        vote_summary = ', '.join(vote_parts)

        # Cluster summaries — key computation and approach per answer group
        cluster_section = self._build_cluster_summaries(detailed_results)

        # Group by method — which libs led to which answers
        method_answers = {}
        for r in detailed_results:
            s = self._summarize_attempt(r)
            method = ', '.join(s['libs'][:3]) if s['libs'] else 'pure reasoning'
            ans = r['Answer']
            if method not in method_answers:
                method_answers[method] = Counter()
            method_answers[method][ans] += 1

        method_lines = []
        for method, counts in method_answers.items():
            parts = [f'{a}x{c}' for a, c in counts.most_common(3)]
            method_lines.append(f'  {method}: {", ".join(parts)}')

        context = (
            f'=== CONTEXT: {len(detailed_results)} previous attempts on this problem ===\n'
            f'Votes: {vote_summary}\n\n'
            + (cluster_section + '\n\n' if cluster_section else '')
            + 'Methods tried → answers:\n'
            + '\n'.join(method_lines) + '\n\n'
            + 'Per-attempt detail:\n'
            + '\n'.join(attempt_lines)
        )

        print(f'Rerun context: {len(context)} chars (~{len(context)//4} tokens)')
        print(context)
        return context
    
    # =========================================================================
    #  GenSelect — Nemotron-7B solution selector (BNB 4-bit, GPU)
    # =========================================================================

    _GENSELECT_PROMPT = (
        'You will be given a challenging math problem followed by {num_solutions} solutions. '
        'Your task is to systematically analyze these solutions to identify the most mathematically sound approach.\n\n'
        'Input Format:\n'
        'Problem: A complex mathematical word problem at advanced high school or college level\n'
        'Solutions: Detailed solutions indexed 0-{max_idx}, each concluding with an answer in \\boxed{{}} notation\n\n'
        'YOUR TASK\n\n'
        'Problem: {problem}\n\n'
        'Solutions:\n{solutions}\n\n'
        'Evaluation Process:\n\n'
        '1. Initial Screening\n'
        '- Group solutions by their final answers\n'
        '- Identify and explain mathematical contradictions between different answers\n'
        '- Eliminate solutions with clear mathematical errors\n\n'
        '2. Detailed Analysis\n'
        'For remaining solutions, evaluate:\n'
        '- Mathematical precision and accuracy\n'
        '- Logical progression of steps\n'
        '- Completeness of mathematical reasoning\n'
        '- Proper use of mathematical notation, including \\boxed{{}}\n'
        '- Handling of edge cases or special conditions\n'
        '- For solutions containing and addressing errors, evaluate the error identification and correction methodology.\n\n'
        '3. Solution Comparison\n'
        'Compare viable solutions based on:\n'
        '- Efficiency of approach\n'
        '- Clarity of mathematical reasoning\n'
        '- Sophistication of method\n'
        '- Robustness of solution (works for all cases)\n\n'
        'Your response should include:\n'
        '1. Brief analysis of conflicting answers\n'
        '2. Detailed evaluation of mathematically sound solutions\n'
        '3. Justification for eliminating incorrect solutions\n'
        '4. Clear explanation for selecting the best approach\n\n'
        'End your evaluation with exactly:\n'
        'Judgment: [IDX]\n'
        'where IDX is the index 0-{max_idx} of the best solution.'
    )

    def _load_genselect_model(self):
        """Lazy-load OpenReasoning-Nemotron-7B (BNB 4-bit) for GenSelect on GPU. Loads once, cached on self."""
        if hasattr(self, '_gs_model'):
            return self._gs_model, self._gs_tokenizer

        import glob as _glob

        # Find model path on Kaggle — models mount at /kaggle/input/<slug>/<framework>/<variation>/<version>/
        gs_candidates = [
            '/kaggle/input/openreasoningnemotron-7b-bnb4/transformers/default/1',
            '/kaggle/input/openreasoningnemotron-7b-bnb4',
            '/kaggle/input/openreasoning-nemotron-7b-bnb4/transformers/default/1',
            '/kaggle/input/openreasoning-nemotron-7b-bnb4',
        ] + sorted(_glob.glob('/kaggle/input/**/openreasoning*nemotron*', recursive=True))

        gs_path = None
        for c in gs_candidates:
            if os.path.isdir(c) and os.path.exists(os.path.join(c, 'config.json')):
                gs_path = c
                break

        if gs_path is None:
            print('GenSelect-7B: model not found at any candidate path — skipping')
            print(f'  Searched: {gs_candidates}')
            self._gs_model = None
            self._gs_tokenizer = None
            return None, None

        print(f'GenSelect-7B: loading model from {gs_path}...')
        gs_start = time.time()

        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        self._gs_tokenizer = AutoTokenizer.from_pretrained(gs_path)
        self._gs_model = AutoModelForCausalLM.from_pretrained(
            gs_path,
            device_map='auto',
            torch_dtype=torch.bfloat16,
        )

        gs_load_time = time.time() - gs_start
        print(f'GenSelect-7B: model loaded in {gs_load_time:.1f}s')

        # GPU memory snapshot
        try:
            if torch.cuda.is_available():
                for i in range(torch.cuda.device_count()):
                    alloc = torch.cuda.memory_allocated(i) / 1e9
                    reserved = torch.cuda.memory_reserved(i) / 1e9
                    total = torch.cuda.get_device_properties(i).total_mem / 1e9
                    print(f'  GPU {i} after GenSelect load: {alloc:.1f}GB alloc / {reserved:.1f}GB reserved / {total:.1f}GB total')
        except Exception as e:
            print(f'  GPU stats unavailable: {e}')

        return self._gs_model, self._gs_tokenizer

    def _summarize_one_attempt(self, problem: str, result: dict) -> dict:
        """Summarize a single R1 attempt via the 120B vLLM server.

        Called in streaming fashion as each R1 attempt finishes. Returns dict with
        answer, summary text, status, and timing. Falls back gracefully on any failure.

        Matches NVIDIA's pipeline: Qwen2.5-32B-Instruct summarizes each solution
        into a clean math writeup (max 2048 tokens). We use our 120B instead.
        """
        import re
        answer = result.get('Answer')
        att_idx = result.get('Attempt', '?')
        summ_start = time.time()

        try:
            # Build raw trace from conversation
            conv = result.get('Conversation', [])
            if not conv:
                print(f'  Summ Att {att_idx}: empty conversation → skip')
                return {'answer': answer, 'summary': f'\\boxed{{{answer}}}', 'status': 'empty',
                        'time': 0.0, 'trace_chars': 0, 'summary_chars': 0}

            trace_parts = []
            for entry in conv:
                if entry.get('type') == 'reasoning':
                    text = entry.get('text', '').strip()
                    if len(text) > 30:
                        trace_parts.append(text)
                elif entry.get('type') == 'code_call':
                    code = entry.get('code', '').strip()
                    output = entry.get('output', '').strip()
                    is_error = entry.get('error', False)
                    if code:
                        trace_parts.append(f'Code:\n{code}')
                    if output:
                        prefix = 'Error' if is_error else 'Output'
                        trace_parts.append(f'{prefix}: {output}')

            trace_text = '\n'.join(trace_parts)
            trace_chars = len(trace_text)

            # Skip very short traces — already clean enough
            if trace_chars < 300:
                print(f'  Summ Att {att_idx}: short trace ({trace_chars} chars) → skip')
                return {'answer': answer, 'summary': trace_text + f'\n\\boxed{{{answer}}}',
                        'status': 'skip-short', 'time': 0.0,
                        'trace_chars': trace_chars, 'summary_chars': trace_chars}

            # Truncate very long traces for the summarizer input (keep first 3000 + last 3000)
            input_trace = trace_text
            was_truncated = False
            if len(input_trace) > 6000:
                input_trace = input_trace[:3000] + '\n[... middle omitted ...]\n' + input_trace[-3000:]
                was_truncated = True

            prompt_text = self._SUMMARIZE_PROMPT.format(problem=problem, trace=input_trace)

            system_content = (
                SystemContent.new()
                .with_model_identity('You are a mathematical solution writer. Write clean, concise solutions.')
                .with_reasoning_effort(reasoning_effort=ReasoningEffort.MEDIUM)
            )
            system_msg = Message.from_role_and_content(Role.SYSTEM, system_content)
            user_msg = Message.from_role_and_content(Role.USER, prompt_text)
            conv_msg = Conversation.from_messages([system_msg, user_msg])
            prompt_ids = self.encoding.render_conversation_for_completion(conv_msg, Role.ASSISTANT)

            response = self.client.completions.create(
                model=self.cfg.served_model_name,
                prompt=prompt_ids,
                max_tokens=2048,
                temperature=0.3,
                stream=False,
                seed=self.cfg.seed + att_idx + 3000,
                extra_body={
                    'min_p': self.cfg.min_p,
                    'stop_token_ids': self.stop_token_ids,
                    'return_token_ids': True,
                }
            )
            summary = response.choices[0].text or ''
            summ_time = time.time() - summ_start

            # Validate: \boxed{answer} must match original
            boxed_matches = re.findall(r'\\boxed\{([^}]+)\}', summary)
            if boxed_matches:
                try:
                    summary_answer = int(boxed_matches[-1])
                    if summary_answer == answer:
                        print(f'  Summ Att {att_idx}: OK | answer={answer} | '
                              f'{trace_chars}→{len(summary)} chars | {summ_time:.1f}s'
                              f'{" (input truncated)" if was_truncated else ""}')
                        return {'answer': answer, 'summary': summary, 'status': 'ok',
                                'time': summ_time, 'trace_chars': trace_chars,
                                'summary_chars': len(summary)}
                except (ValueError, TypeError):
                    pass

            # Answer mismatch or no boxed — fall back to raw trace with smart truncation
            fallback = trace_text[:3000] + f'\n\\boxed{{{answer}}}'
            found_ans = boxed_matches[-1] if boxed_matches else 'none'
            print(f'  Summ Att {att_idx}: MISMATCH | expected={answer} found={found_ans} | '
                  f'{trace_chars} chars | {summ_time:.1f}s → using raw fallback')
            return {'answer': answer, 'summary': fallback,
                    'status': 'answer-mismatch', 'time': summ_time,
                    'trace_chars': trace_chars, 'summary_chars': len(fallback)}

        except Exception as e:
            summ_time = time.time() - summ_start
            print(f'  Summ Att {att_idx}: ERROR | {type(e).__name__}: {e} | {summ_time:.1f}s')
            return {'answer': answer, 'summary': f'\\boxed{{{answer}}}',
                    'status': f'error-{type(e).__name__}', 'time': summ_time,
                    'trace_chars': 0, 'summary_chars': 0}

    # NVIDIA's solution summarization prompt (from NeMo-Skills repo: summarize-solution.yaml)
    _SUMMARIZE_PROMPT = (
        'I will give you a math problem and a computational solution trace that explores '
        'different approaches, runs code, makes mistakes along the way, corrects them, '
        'and eventually arrives at the right answer. Your task is to write a clean version '
        'of the final correct solution without all the exploration. Cover all the details '
        'of the final solution including key computations and their results.\n\n'
        'Problem:\n{problem}\n\n'
        'Solution trace:\n{trace}\n\n'
        'Now write a clean version of the final correct solution without all the exploration '
        'but cover all the details of the final solution. End with the answer in \\boxed{{}} notation.'
    )

    def _build_genselect_solutions(self, problem: str, detailed_results: list, use_summaries: bool = True):
        """Build solution summaries for GenSelect judges.

        Aligned with NVIDIA GenSelect papers (arXiv 2507.17797, 2504.16891):
        - No metadata headers, no vote counts — judge evaluates math only
        - Uses LLM-generated summaries when available (streamed during R1)
        - Per-cluster: picks the LONGEST valid summary (NVIDIA's exact criterion)
        - Falls back to smart truncation of raw traces when summaries unavailable
        - Both judges (7B + 120B) share the same solution text

        Returns (solutions_text, answer_map, num_solutions).
        """
        from collections import Counter

        # Group by answer, skip None
        clusters = {}
        for r in detailed_results:
            ans = r['Answer']
            if ans is not None:
                if ans not in clusters:
                    clusters[ans] = []
                clusters[ans].append(r)

        if len(clusters) < 2:
            return None, None, 0

        # Sort by vote count descending, take top 8 (prompt fits ~16 solutions per GenSelect paper)
        sorted_clusters = sorted(clusters.items(), key=lambda x: -len(x[1]))[:8]

        total_valid = sum(len(v) for v in clusters.values())
        none_count = sum(1 for r in detailed_results if r['Answer'] is None)
        print(f'\n=== GENSELECT SOLUTIONS ({len(clusters)} clusters, {total_valid} valid answers, {none_count} Nones) ===')

        # Get cached streaming summaries (populated during R1)
        cached = getattr(self, '_cached_summaries', {})
        n_cached = len(cached)
        n_ok = sum(1 for v in cached.values() if v.get('status') == 'ok')
        if use_summaries and n_cached > 0:
            print(f'  Cached summaries: {n_cached} total, {n_ok} clean (status=ok)')

        solutions = []
        answer_map = {}  # idx → answer value

        for ans, atts in sorted_clusters:
            sol_text = None
            sol_source = 'none'

            # === Step 1: Try to use cached streaming summaries (NVIDIA's approach) ===
            # For each cluster, find all cached summaries with matching answer,
            # pick the LONGEST valid one (NVIDIA's exact criterion: longest post-</think> text).
            if use_summaries and cached:
                matching_summaries = []
                for r in atts:
                    att_idx = r.get('Attempt')
                    if att_idx in cached:
                        entry = cached[att_idx]
                        if entry.get('status') == 'ok' and entry.get('answer') == ans:
                            matching_summaries.append(entry)

                if matching_summaries:
                    # NVIDIA criterion: pick the longest valid summary
                    best_summary = max(matching_summaries, key=lambda e: len(e.get('summary', '')))
                    sol_text = best_summary['summary']
                    # Ensure it ends with \boxed{answer}
                    if f'\\boxed{{{ans}}}' not in sol_text:
                        sol_text += f'\n\\boxed{{{ans}}}'
                    sol_source = f'summary(best of {len(matching_summaries)})'

            # === Step 2: Fall back to raw trace with smart truncation ===
            if sol_text is None:
                sol_text = self._build_raw_trace_solution(atts, ans)
                sol_source = 'raw-trace'

            idx = len(solutions)
            answer_map[idx] = ans
            solutions.append(f'Solution {idx}: {sol_text}')
            # Preview: first 200 chars of solution text
            preview = sol_text[:200].replace('\n', ' ')
            print(f'  Sol {idx}: answer={ans}, votes={len(atts)}, '
                  f'source={sol_source}, chars={len(sol_text)}')
            print(f'    Preview: {preview}...')

        solutions_text = '\n\n'.join(solutions)
        print(f'Total: {len(solutions)} solutions, {len(solutions_text)} chars')
        print(f'=== END GENSELECT SOLUTIONS ===\n')
        return solutions_text, answer_map, len(solutions)

    def _build_raw_trace_solution(self, atts: list, ans: int) -> str:
        """Build a solution string from raw R1 traces with smart truncation.

        Used as fallback when LLM summaries are unavailable. Picks the best
        representative attempt, strips boilerplate, and applies smart truncation.
        """
        MAX_SOL_CHARS = 6000

        # Representative selection: prefer moderate depth (3-15 code turns),
        # then lower entropy. Avoids 0-turn empty attempts AND 60-turn monsters.
        atts_with_content = [r for r in atts if len(r.get('Conversation', [])) > 0]
        pool = atts_with_content if atts_with_content else atts

        def _rep_score(r):
            conv = r.get('Conversation', [])
            code_turns = sum(1 for e in conv if e.get('type') == 'code_call')
            entropy = r.get('Entropy', float('inf'))
            if not math.isfinite(entropy):
                entropy = 999
            capped_turns = min(code_turns, 15)
            return (capped_turns, -entropy)

        best = max(pool, key=_rep_score)
        conv = best.get('Conversation', [])

        # === Build turn list, strip boilerplate ===
        turns = []
        for entry in conv:
            if entry.get('type') == 'reasoning':
                text = entry.get('text', '').strip()
                if len(text) > 30:
                    turns.append({'type': 'reasoning', 'text': text})
            elif entry.get('type') == 'code_call':
                code = entry.get('code', '').strip()
                output = entry.get('output', '').strip()
                is_error = entry.get('error', False)
                if not is_error and output in ('', '[WARN] No output. Use print() to see results.'):
                    lines = [l.strip() for l in code.split('\n') if l.strip()]
                    if all(l.startswith(('import ', 'from ')) for l in lines):
                        continue
                turns.append({'type': 'code', 'code': code, 'output': output, 'is_error': is_error})

        # === Format turns cleanly ===
        def _format_turn(t):
            if t['type'] == 'reasoning':
                return t['text']
            code = t.get('code', '')
            output = t.get('output', '')
            is_err = t.get('is_error', False)
            parts = []
            if code:
                clean_lines = []
                for line in code.split('\n'):
                    stripped = line.strip()
                    if stripped.startswith(('import ', 'from ')) and len(stripped) > 80:
                        continue
                    clean_lines.append(line)
                clean_code = '\n'.join(clean_lines).strip()
                if clean_code:
                    parts.append(clean_code)
            if output:
                if len(output) > 500:
                    output = output[:250] + ' ... ' + output[-200:]
                prefix = 'Error' if is_err else 'Output'
                parts.append(f'{prefix}: {output}')
            return '\n'.join(parts)

        formatted_turns = [_format_turn(t) for t in turns]
        formatted_turns = [ft for ft in formatted_turns if ft.strip()]

        full_text = '\n\n'.join(formatted_turns)

        if len(full_text) <= MAX_SOL_CHARS:
            sol_text = full_text
        else:
            # Smart truncation: keep first turn, last 2 turns, and KEY middle turns
            keep_indices = set()
            if formatted_turns:
                keep_indices.add(0)
            if len(formatted_turns) >= 2:
                keep_indices.add(len(formatted_turns) - 1)
                keep_indices.add(len(formatted_turns) - 2)

            for i, t in enumerate(turns):
                if i in keep_indices:
                    continue
                if t['type'] == 'code':
                    code_lower = t.get('code', '').lower()
                    if t.get('is_error'):
                        keep_indices.add(i)
                    elif i > 0 and turns[i - 1].get('is_error'):
                        keep_indices.add(i)
                    elif any(kw in code_lower for kw in ['assert', 'check', 'verify', 'validate']):
                        keep_indices.add(i)
                    elif any(kw in code_lower for kw in ['from scipy.optimize', 'from pulp',
                                'milp(', 'linprog(', 'lru_cache', 'backtrack', 'convex']):
                        keep_indices.add(i)
                elif t['type'] == 'reasoning' and len(t.get('text', '')) > 100:
                    keep_indices.add(i)

            kept_parts = []
            budget_remaining = MAX_SOL_CHARS - 100
            sorted_keep = sorted(keep_indices)
            prev_idx = -1

            for ki in sorted_keep:
                if ki >= len(formatted_turns):
                    continue
                turn_text = formatted_turns[ki]
                if len(turn_text) > 1500:
                    turn_text = turn_text[:500] + '\n... (trimmed) ...\n' + turn_text[-800:]
                if len(turn_text) > budget_remaining:
                    break
                if ki > prev_idx + 1 and prev_idx >= 0:
                    gap = ki - prev_idx - 1
                    gap_marker = f'[... {gap} step{"s" if gap > 1 else ""} omitted ...]'
                    kept_parts.append(gap_marker)
                    budget_remaining -= len(gap_marker) + 2
                kept_parts.append(turn_text)
                budget_remaining -= len(turn_text) + 2
                prev_idx = ki

            if sorted_keep and prev_idx < len(formatted_turns) - 1:
                remaining = len(formatted_turns) - 1 - prev_idx
                kept_parts.append(f'[... {remaining} more step{"s" if remaining > 1 else ""} to final answer ...]')

            sol_text = '\n\n'.join(kept_parts)

        sol_text += f'\n\\boxed{{{ans}}}'
        return sol_text

    def _parse_sol_entries(self, solutions_text: str, num_solutions: int) -> list:
        """Parse 'Solution N: ...' text into list of (original_idx, body) tuples for permutation."""
        sol_entries = []
        for idx in range(num_solutions):
            prefix = f'Solution {idx}: '
            start = solutions_text.find(prefix)
            if start < 0:
                continue
            end = len(solutions_text)
            for next_idx in range(idx + 1, num_solutions):
                next_start = solutions_text.find(f'Solution {next_idx}: ')
                if next_start > 0:
                    end = next_start
                    break
            body = solutions_text[start + len(prefix):end].strip()
            sol_entries.append((idx, body))
        return sol_entries

    def _genselect_permute_and_judge(self, sol_entries, answer_map, problem, num_solutions,
                                      model_fn, label, gs_passes, gs_budget):
        """Core GenSelect loop: permute solutions, judge, collect votes.

        model_fn(prompt) -> response_text. Called for each pass.
        Returns (answer, reasoning) or (None, None).
        """
        import re
        import random as _random
        from collections import Counter

        gs_start = time.time()
        judgments = []
        best_reasoning = ''

        for gs_i in range(gs_passes):
            if time.time() - gs_start > gs_budget - 15:
                print(f'  {label} time limit — stopping after {gs_i} passes')
                break

            try:
                pass_start = time.time()
                perm = list(range(len(sol_entries)))
                _random.shuffle(perm)

                permuted_sols = []
                perm_to_orig = {}
                for new_idx, old_pos in enumerate(perm):
                    orig_idx, body = sol_entries[old_pos]
                    perm_to_orig[new_idx] = orig_idx
                    permuted_sols.append(f'Solution {new_idx}: {body}')

                permuted_text = '\n\n'.join(permuted_sols)

                prompt = self._GENSELECT_PROMPT.format(
                    problem=problem,
                    solutions=permuted_text,
                    num_solutions=num_solutions,
                    max_idx=num_solutions - 1
                )

                # Log full prompt on first pass for validation
                if gs_i == 0:
                    print(f'\n  --- {label} FULL PROMPT (pass 0, {len(prompt)} chars) ---')
                    print(prompt)
                    print(f'  --- END {label} PROMPT ---\n')

                response = model_fn(prompt, gs_i)
                pass_time = time.time() - pass_start

                # Map permuted indices back to original for logging
                perm_answer_order = [answer_map.get(perm_to_orig.get(i), '?') for i in range(len(perm))]

                matches = re.findall(r'Judg[e]?ment:\s*(\d+)', response)
                if matches:
                    perm_idx = int(matches[-1])
                    if perm_idx in perm_to_orig:
                        orig_idx = perm_to_orig[perm_idx]
                        if orig_idx in answer_map:
                            judgments.append(answer_map[orig_idx])
                            print(f'  {label} Pass {gs_i}: answer={answer_map[orig_idx]} | '
                                  f'perm_order={perm_answer_order} | '
                                  f'prompt={len(prompt)}c response={len(response)}c | {pass_time:.1f}s')
                            if not best_reasoning:
                                judgment_pos = response.rfind('Judgment')
                                if judgment_pos > 0:
                                    best_reasoning = response[max(0, judgment_pos - 500):judgment_pos].strip()
                        else:
                            print(f'  {label} Pass {gs_i}: orig index {orig_idx} not in answer_map | {pass_time:.1f}s')
                    else:
                        print(f'  {label} Pass {gs_i}: permuted index {perm_idx} out of range (max={len(perm)-1}) | {pass_time:.1f}s')
                else:
                    # Log a snippet of response to help debug extraction failures
                    snippet = response[-200:].replace('\n', ' ') if response else '(empty)'
                    print(f'  {label} Pass {gs_i}: no judgment extracted | response_tail: {snippet} | {pass_time:.1f}s')

            except Exception as e:
                print(f'  {label} Pass {gs_i}: error — {type(e).__name__}: {e}')

        gs_elapsed = time.time() - gs_start
        print(f'{label} completed {len(judgments)}/{gs_passes} valid passes in {gs_elapsed:.1f}s')

        if not judgments:
            print(f'{label}: no valid judgments')
            return None, None

        gs_votes = Counter(judgments)
        gs_winner, gs_count = gs_votes.most_common(1)[0]
        print(f'{label} votes: {dict(gs_votes.most_common())}')
        print(f'{label} winner: {gs_winner} ({gs_count}/{len(judgments)} votes)')
        if best_reasoning:
            print(f'{label} reasoning: {best_reasoning[:300]}...')

        if gs_count > len(judgments) / 2:
            return gs_winner, best_reasoning
        else:
            print(f'{label}: no majority — inconclusive')
            return None, None

    def _run_genselect(self, problem: str, detailed_results: list):
        """Run GenSelect using Nemotron-7B on GPU (trained judge, 8 passes)."""

        solutions_text, answer_map, num_solutions = self._build_genselect_solutions(problem, detailed_results)
        if solutions_text is None:
            print('GenSelect-7B: fewer than 2 competing answers — skipping')
            return None, None

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        reserved = max(0, self.problems_remaining) * self.cfg.reserved_per_problem
        gs_budget = min(time_left - reserved, 180)
        if gs_budget < 30:
            print(f'GenSelect-7B skipped — insufficient budget ({gs_budget:.0f}s)')
            return None, None

        model, tokenizer = self._load_genselect_model()
        if model is None:
            return None, None

        sol_entries = self._parse_sol_entries(solutions_text, num_solutions)
        if len(sol_entries) < 2:
            print('GenSelect-7B: could not parse solutions')
            return None, None

        print(f'\n=== GENSELECT-7B: {num_solutions} candidates, budget={gs_budget:.0f}s ===')
        print(f'Candidates: {", ".join(f"Sol {i}={answer_map[i]}" for i in range(num_solutions))}')

        import torch

        def model_fn_7b(prompt, pass_idx):
            chat = [{'role': 'user', 'content': prompt}]
            input_ids = tokenizer.apply_chat_template(chat, return_tensors='pt')
            with torch.no_grad():
                outputs = model.generate(
                    input_ids.to(model.device),
                    max_new_tokens=8192,
                    temperature=0.6,
                    top_p=0.95,
                    do_sample=True,
                    use_cache=True,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            return tokenizer.decode(outputs[0], skip_special_tokens=True)

        return self._genselect_permute_and_judge(
            sol_entries, answer_map, problem, num_solutions,
            model_fn_7b, 'GenSelect-7B', gs_passes=8, gs_budget=gs_budget
        )

    def _run_genselect_vllm(self, problem: str, detailed_results: list):
        """Run GenSelect using 120B model via vLLM (zero-shot judge, 8 passes).

        Uses cached streaming summaries (generated during R1) for clean math solutions.
        Both judges (7B + 120B) share the same solution text.
        """

        solutions_text, answer_map, num_solutions = self._build_genselect_solutions(
            problem, detailed_results
        )
        if solutions_text is None:
            print('GenSelect-120B: fewer than 2 competing answers — skipping')
            return None, None

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        reserved = max(0, self.problems_remaining) * self.cfg.reserved_per_problem
        gs_budget = min(time_left - reserved, 180)
        if gs_budget < 30:
            print(f'GenSelect-120B skipped — insufficient budget ({gs_budget:.0f}s)')
            return None, None

        sol_entries = self._parse_sol_entries(solutions_text, num_solutions)
        if len(sol_entries) < 2:
            print('GenSelect-120B: could not parse solutions')
            return None, None

        print(f'\n=== GENSELECT-120B: {num_solutions} candidates, budget={gs_budget:.0f}s ===')
        print(f'Candidates: {", ".join(f"Sol {i}={answer_map[i]}" for i in range(num_solutions))}')

        gs_system = 'You are an expert IMO-level mathematical judge. Evaluate the competing solutions carefully and identify the correct one.'

        def model_fn_vllm(prompt, pass_idx):
            # Encode via Harmony: system (no tools) + user message → token IDs
            system_content = (
                SystemContent.new()
                .with_model_identity(gs_system)
                .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            )
            system_msg = Message.from_role_and_content(Role.SYSTEM, system_content)
            user_msg = Message.from_role_and_content(Role.USER, prompt)
            conv = Conversation.from_messages([system_msg, user_msg])
            prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)

            response = self.client.completions.create(
                model=self.cfg.served_model_name,
                prompt=prompt_ids,
                max_tokens=8192,
                temperature=0.6,
                stream=False,
                seed=self.cfg.seed + pass_idx + 1000,
                extra_body={
                    'min_p': self.cfg.min_p,
                    'stop_token_ids': self.stop_token_ids,
                    'return_token_ids': True,
                }
            )
            return response.choices[0].text or ''

        return self._genselect_permute_and_judge(
            sol_entries, answer_map, problem, num_solutions,
            model_fn_vllm, 'GenSelect-120B', gs_passes=8, gs_budget=gs_budget
        )

    def print_gpu_summary(self):
        """Print GPU memory and vLLM server stats."""
        try:
            import torch
            if torch.cuda.is_available():
                for i in range(torch.cuda.device_count()):
                    alloc = torch.cuda.memory_allocated(i) / 1e9
                    reserved = torch.cuda.memory_reserved(i) / 1e9
                    total = torch.cuda.get_device_properties(i).total_mem / 1e9
                    print(f'  GPU {i}: {alloc:.1f}GB alloc / {reserved:.1f}GB reserved / {total:.1f}GB total')
        except Exception as e:
            print(f'  torch GPU stats unavailable: {e}')

        # vLLM server metrics if available
        try:
            import requests
            resp = requests.get(f'{self.base_url.replace("/v1","")}/metrics', timeout=5)
            if resp.ok:
                lines = resp.text.strip().split('\n')
                for line in lines:
                    if any(k in line for k in ['gpu_cache_usage', 'num_requests', 'avg_generation_throughput']) and not line.startswith('#'):
                        print(f'  {line}')
        except Exception:
            pass

    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass


In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item()
    question_text = question.item()
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    # Solve 3 dummy competition test questions → produces submission.parquet
    # Required for Kaggle to accept this notebook version for submission selection.
    # Output goes to normal Kaggle logs (cell 18 redirects to diagnostic.log later).
    # Cell 18 resets solver clocks after this, so time spent here doesn't affect test budget.
    test_csv_candidates = [
        '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',
        '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',
    ]
    test_csv = next((p for p in test_csv_candidates if os.path.isfile(p)), test_csv_candidates[0])
    print(f'Using test CSV: {test_csv}')
    inference_server.run_local_gateway((test_csv,))


In [ ]:
import gc
# =============================================================================
# TEST FRAMEWORK — Full 50-problem run
# =============================================================================
# Diagnostic output is written to /kaggle/working/diagnostic.log (persists as kernel output)

import sys
import os

class TeeLogger:
    """Write to log file only. Redirects at fd level to prevent Kaggle UI leakage."""
    def __init__(self, filepath='/kaggle/working/diagnostic.log', stream='stdout'):
        self.log = open(filepath, 'w', buffering=1)  # line-buffered for live output
        if stream == 'stdout':
            self._orig_fd = os.dup(1)  # save original fd 1
            os.dup2(self.log.fileno(), 1)  # point fd 1 → log file
        elif stream == 'stderr':
            self._orig_fd = os.dup(2)
            os.dup2(self.log.fileno(), 2)
    def write(self, message):
        self.log.write(message)
    def flush(self):
        self.log.flush()

sys.stdout = TeeLogger('/kaggle/working/diagnostic.log', stream='stdout')
sys.stderr = TeeLogger('/kaggle/working/diagnostic_stderr.log', stream='stderr')

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):

    def _print_attempt_full(r, expected):
        """Print complete diagnostic for one attempt — every step, no truncation."""
        attempt = r.get('Attempt', '?')
        answer = r.get('Answer', None)
        entropy = r.get('Entropy', float('inf'))
        py_calls = r.get('Python Calls', 0)
        py_errors = r.get('Python Errors', 0)
        resp_len = r.get('Response Length', 0)
        temp = r.get('Temperature', '?')
        elapsed = r.get('Time', '?')
        confidence = r.get('Confidence', 0.0)
        source = r.get('Source', '?')
        stop_reason = r.get('Stop_Reason', '?')
        time_remaining = r.get('Time_Remaining', '?')
        vboxed_cps = r.get('Vboxed_Checkpoints', [])
        bells_data = r.get('Bells', {})
        status = 'CORRECT' if answer == expected else ('WRONG' if answer is not None else 'NONE')
        print(f'\n    --- Attempt {attempt} [{status}] answer={answer} expected={expected} temp={temp} entropy={entropy:.3f} ---')
        print(f'        python_calls={py_calls} errors={py_errors} tokens={resp_len} time={elapsed}s time_left={time_remaining}s')
        print(f'        source={source} confidence={confidence} stop={stop_reason}')
        if vboxed_cps:
            cp_str = ', '.join(f'turn{cp["turn"]}={cp["answer"]}' for cp in vboxed_cps)
            print(f'        vboxed_checkpoints: {cp_str}')
        if any(v > 0 for v in bells_data.values()):
            print(f'        bells: {bells_data}')

        conv = r.get('Conversation', [])
        if conv:
            imports_found = set()
            for entry in conv:
                if entry.get('type') == 'code_call':
                    code = entry.get('code', '')
                    for line in code.split('\n'):
                        line = line.strip()
                        if line.startswith('import ') or line.startswith('from '):
                            imports_found.add(line)
            if imports_found:
                print(f'        Libraries: {", ".join(sorted(imports_found))}')

            for entry in conv:
                turn = entry.get('turn', '?')
                etype = entry.get('type', '?')
                if etype == 'reasoning':
                    text = entry.get('text', '')
                    print(f'\n        [Turn {turn} REASONING] ({len(text)} chars)')
                    print(f'        {text}')
                elif etype == 'code_call':
                    code = entry.get('code', '')
                    output = entry.get('output', '')
                    is_error = entry.get('error', False)
                    err_marker = ' [ERROR]' if is_error else ''
                    print(f'\n        [Turn {turn} CODE{err_marker}]')
                    print(f'        >>> Code:')
                    for cline in code.split('\n'):
                        print(f'            {cline}')
                    print(f'        >>> Output:')
                    for oline in output.split('\n')[:50]:
                        print(f'            {oline}')
                    if len(output.split(chr(10))) > 50:
                        print(f'            ... ({len(output.split(chr(10)))} lines total)')

    def _get_gpu_stats():
        """Get GPU memory stats if available."""
        try:
            import torch
            if torch.cuda.is_available():
                alloc = torch.cuda.memory_allocated() / 1e9
                reserved = torch.cuda.memory_reserved() / 1e9
                return f'GPU mem: {alloc:.1f}GB alloc / {reserved:.1f}GB reserved'
        except Exception:
            pass
        return 'GPU stats unavailable'

    def run_test_batch(name, problems, solver):
        """Run a batch of test problems with full diagnostics."""
        print(f'\n{"="*80}')
        print(f'  BATCH: {name} ({len(problems)} problems)')
        print(f'  {_get_gpu_stats()}')
        print(f'{"="*80}')

        batch_correct = 0
        batch_total = len(problems)
        batch_results = []
        batch_start = time.time()

        for idx, p in enumerate(problems):
            pid = p['id']
            expected = p['answer']
            problem_text = p['problem']

            print(f'\n{"─"*70}')
            print(f'  [{idx+1}/{batch_total}] Problem {pid} | Expected: {expected}')
            print(f'{"─"*70}')

            prob_start = time.time()
            predicted = solver.solve_problem(problem_text)
            prob_elapsed = time.time() - prob_start

            correct = (predicted == expected)
            batch_correct += int(correct)
            status = 'CORRECT' if correct else 'WRONG'

            result_entry = {
                'id': pid,
                'predicted': predicted,
                'expected': expected,
                'correct': correct,
                'time': round(prob_elapsed, 1),
            }

            detailed = getattr(solver, '_last_detailed_results', [])
            if detailed:
                answers = [r['Answer'] for r in detailed if r.get('Answer') is not None]
                from collections import Counter
                vote_dist = dict(Counter(answers).most_common())
                result_entry['votes'] = vote_dist
                result_entry['num_answers'] = len(answers)
                result_entry['num_none'] = sum(1 for r in detailed if r.get('Answer') is None)
                result_entry['num_errors'] = sum(r.get('Python Errors', 0) for r in detailed)

                # Capture GenSelect + summarization metadata
                meta = getattr(solver, '_last_problem_meta', {})
                if meta:
                    result_entry['meta'] = meta

                print(f'\n  STATUS: {status} | Predicted: {predicted} | Expected: {expected} | Time: {prob_elapsed:.1f}s')
                print(f'  Votes: {vote_dist} | Answers: {len(answers)} | Nones: {result_entry["num_none"]} | Errors: {result_entry["num_errors"]}')

                if not correct:
                    print(f'\n  WRONG ANSWER ANALYSIS:')
                    for r in detailed:
                        if r.get('Answer') is not None and r['Answer'] != expected:
                            print(f'    Attempt {r["Attempt"]}: answered {r["Answer"]} (entropy={r["Entropy"]:.3f})')
                    correct_attempts = [r for r in detailed if r.get('Answer') == expected]
                    if correct_attempts:
                        print(f'    {len(correct_attempts)} attempt(s) got the right answer but were outvoted')

                for r in detailed:
                    _print_attempt_full(r, expected)
            else:
                print(f'\n  STATUS: {status} | Predicted: {predicted} | Expected: {expected} | Time: {prob_elapsed:.1f}s')

            batch_results.append(result_entry)
            gc.collect()

            running_pct = batch_correct / (idx + 1) * 100
            print(f'\n  Running: {batch_correct}/{idx+1} ({running_pct:.0f}%) | Elapsed: {time.time()-batch_start:.0f}s')

        batch_elapsed = time.time() - batch_start
        print(f'\n{"="*80}')
        print(f'  BATCH COMPLETE: {name}')
        print(f'  Score: {batch_correct}/{batch_total} ({batch_correct/max(batch_total,1)*100:.0f}%)')
        print(f'  Time: {batch_elapsed:.0f}s')
        if batch_results:
            times = [r['time'] for r in batch_results]
            print(f'  Timing: min={min(times):.0f}s  max={max(times):.0f}s  median={sorted(times)[len(times)//2]:.0f}s')
        print(f'{"="*80}')

        return batch_correct, batch_total, batch_results

    def load_csv_problems(problems_csv, answers_csv):
        """Load all problems from CSV files."""
        import csv
        problems = {}
        answers = {}
        with open(problems_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                problems[row['id']] = row['problem']
        with open(answers_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                answers[row['id']] = int(row['answer'])
        return [{'id': pid, 'problem': problems[pid], 'answer': answers[pid]}
                for pid in problems if pid in answers]

    # =========================================================================
    #  LOAD TEST SET — pick: set_a (50), set_b (50), set_c (4), set_d (1)
    # =========================================================================

    TEST_SET = 'set_c'
    import glob as _glob
    _test_data_candidates = [
        f'/kaggle/input/aimo3-test-data/{TEST_SET}_problems.csv',
    ] + sorted(_glob.glob(f'/kaggle/input/**/{TEST_SET}_problems.csv', recursive=True))
    problems_csv = next((p for p in _test_data_candidates if os.path.isfile(p)), _test_data_candidates[0])
    answers_csv = problems_csv.replace('_problems.csv', '_answers.csv')

    all_problems = load_csv_problems(problems_csv, answers_csv)

    import random
    random.seed(42)
    random.shuffle(all_problems)

    solver.problems_remaining = len(all_problems)
    solver.notebook_start_time = time.time()  # reset clock — cell 17 gateway may have consumed time

    print(f'\n=== PROBLEM MANIFEST ===')
    print(f'  Test set: {TEST_SET} | Total: {len(all_problems)} problems')
    print(f'  IDs: {[p["id"] for p in all_problems]}')
    print(f'=== END MANIFEST ===')

    suite_start = time.time()
    total_correct, total_problems, all_results = run_test_batch(
        f'{TEST_SET.upper()} — Wave 2 only', all_problems, solver
    )

    # =========================================================================
    #  GPU METRICS SUMMARY
    # =========================================================================
    try:
        solver.print_gpu_summary()
    except Exception as e:
        print(f'GPU metrics unavailable: {e}')

    # =========================================================================
    #  FINAL SUMMARY
    # =========================================================================
    suite_elapsed = time.time() - suite_start
    print(f'\n{"="*80}')
    print(f'  FINAL SUMMARY')
    print(f'{"="*80}')
    print(f'  Score: {total_correct}/{total_problems} ({total_correct/max(total_problems,1)*100:.1f}%)')
    print(f'  Total time: {suite_elapsed:.0f}s ({suite_elapsed/60:.1f} min)')

    if all_results:
        print(f'\n  {"ID":<8} {"Status":<8} {"Pred":>8} {"Exp":>8} {"Votes":<24} {"Ans":<6} {"Errs":<5} {"Time":>6}')
        print(f'  {"---":<8} {"---":<8} {"---":>8} {"---":>8} {"---":<24} {"---":<6} {"---":<5} {"---":>6}')
        for r in all_results:
            status = 'OK' if r['correct'] else 'FAIL'
            votes_str = str(r.get('votes', {}))
            if len(votes_str) > 22:
                votes_str = votes_str[:22] + '..'
            n_ans = r.get('num_answers', '?')
            n_err = r.get('num_errors', '?')
            print(f'  {r["id"]:<8} {status:<8} {r["predicted"]:>8} {r["expected"]:>8} {votes_str:<24} {str(n_ans):<6} {str(n_err):<5} {r["time"]:>5.0f}s')

        wrong_results = [r for r in all_results if not r['correct']]
        correct_results = [r for r in all_results if r['correct']]
        if wrong_results:
            print(f'\n  WRONG ANSWERS ({len(wrong_results)}):')
            for r in wrong_results:
                print(f'    {r["id"]}: predicted {r["predicted"]}, expected {r["expected"]}, votes={r.get("votes", "?")}')
        if correct_results:
            avg_time = sum(r['time'] for r in correct_results) / len(correct_results)
            print(f'\n  Correct answers avg time: {avg_time:.0f}s')

    print(f'\n{"="*80}')

    # =========================================================================
    #  WRITE QUICK SUMMARY FILE (separate from diagnostic.log)
    # =========================================================================
    try:
        with open('/kaggle/working/summary.txt', 'w') as sf:
            sf.write(f'AIMO3 Run Summary\n{"="*60}\n')
            sf.write(f'Score: {total_correct}/{total_problems} ({total_correct/max(total_problems,1)*100:.1f}%)\n')
            sf.write(f'Time: {suite_elapsed:.0f}s ({suite_elapsed/60:.1f} min)\n')
            sf.write(f'Test set: {TEST_SET}\n\n')

            if all_results:
                # === Aggregate stats ===
                n_nones = sum(r.get('num_none', 0) for r in all_results)
                n_total_ans = sum(r.get('num_answers', 0) for r in all_results)
                n_errors = sum(r.get('num_errors', 0) for r in all_results)
                total_att = n_nones + n_total_ans
                none_pct = n_nones / max(total_att, 1) * 100
                times = [r['time'] for r in all_results]

                sf.write(f'Attempts: {n_total_ans}/{total_att} answered ({100-none_pct:.0f}%) | Nones: {n_nones} | Errors: {n_errors}\n')
                sf.write(f'Timing: min={min(times):.0f}s  max={max(times):.0f}s  median={sorted(times)[len(times)//2]:.0f}s\n')

                # Summarization aggregate
                metas = [r.get('meta', {}) for r in all_results if r.get('meta')]
                if metas:
                    total_summ = sum(m.get('summ_total', 0) for m in metas)
                    total_summ_ok = sum(m.get('summ_ok', 0) for m in metas)
                    total_summ_mm = sum(m.get('summ_mismatch', 0) for m in metas)
                    total_summ_time = sum(m.get('summ_time', 0) for m in metas)
                    n_reruns = sum(1 for m in metas if m.get('rerun'))
                    sf.write(f'Summaries: {total_summ_ok}/{total_summ} clean, {total_summ_mm} mismatch, {total_summ_time:.0f}s total\n')
                    sf.write(f'Reruns: {n_reruns}/{len(metas)}\n')

                sf.write(f'\n{"="*60}\n')
                sf.write(f'PER-PROBLEM DETAIL\n')
                sf.write(f'{"="*60}\n\n')

                for r in all_results:
                    status = 'OK' if r['correct'] else 'FAIL'
                    m = r.get('meta', {})
                    sf.write(f'--- {r["id"]} [{status}] ---\n')
                    sf.write(f'  Predicted: {r["predicted"]}  Expected: {r["expected"]}  Time: {r["time"]:.0f}s\n')

                    # R1 voting
                    votes_str = str(r.get('votes', {}))
                    sf.write(f'  R1 Votes: {votes_str}\n')
                    sf.write(f'  R1 Answers: {r.get("num_answers","?")}  Nones: {r.get("num_none","?")}  Errors: {r.get("num_errors","?")}\n')

                    if m:
                        # Summarization
                        sf.write(f'  Summaries: {m.get("summ_ok",0)}/{m.get("summ_total",0)} clean'
                                 f', {m.get("summ_mismatch",0)} mismatch'
                                 f', {m.get("summ_time",0):.1f}s\n')

                        # GenSelect round 1
                        sf.write(f'  GS Round 1: 7B={m.get("gs7b","—")}  120B={m.get("gs120b","—")}'
                                 f'  majority={m.get("r1_answer","?")}(v{m.get("r1_votes","?")})'
                                 f'  clusters={m.get("r1_clusters","?")}'
                                 f'  time={m.get("gs_time",0):.0f}s\n')

                        # Rerun
                        if m.get('rerun'):
                            sf.write(f'  RERUN: {m.get("rerun_atts",0)} attempts\n')
                            sf.write(f'  GS Round 2: 7B={m.get("gs2_7b","—")}  120B={m.get("gs2_120b","—")}\n')

                        # Decision path
                        sf.write(f'  Path: {m.get("path","?")}\n')
                    sf.write('\n')

                # Wrong answer summary
                wrong = [r for r in all_results if not r['correct']]
                if wrong:
                    sf.write(f'{"="*60}\n')
                    sf.write(f'WRONG ANSWERS ({len(wrong)})\n')
                    sf.write(f'{"="*60}\n')
                    for r in wrong:
                        m = r.get('meta', {})
                        sf.write(f'  {r["id"]}: predicted={r["predicted"]} expected={r["expected"]}'
                                 f'  votes={r.get("votes",{})}'
                                 f'  7B={m.get("gs7b","—")} 120B={m.get("gs120b","—")}'
                                 f'  rerun={m.get("rerun",False)}\n')

    except Exception as e:
        print(f'Summary file write failed: {e}')
